<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **SpaceX  Falcon 9 first stage Landing Prediction**


# Lab 1: Collecting the data


Estimated time needed: **45** minutes


In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch. In this lab, you will collect and make sure the data is in the correct format from an API. The following is an example of a successful and launch.


## Reproduce the historical launch table
Use the IBM archive of the SpaceX v4 launches response. The original live enrichment calls returned HTTP 522 and JSON decode errors. Join the historical enrichment fields from IBM’s dated course extract. See the README for input URLs.

In [1]:
from pathlib import Path
import json
import pandas as pd

raw = pd.DataFrame(json.loads(Path("data/launches.json").read_text()))
print(f"{len(raw)} launch records in the archive")
raw[["name", "date_utc", "rocket"]].head()

107 launch records in the archive
Out[0]: 
          name                  date_utc                    rocket
0    FalconSat  2006-03-24T22:30:00.000Z  5e9d0d95eda69955f709d1eb
1      DemoSat  2007-03-21T01:10:00.000Z  5e9d0d95eda69955f709d1eb
2  Trailblazer  2008-08-03T03:34:00.000Z  5e9d0d95eda69955f709d1eb
3       RatSat  2008-09-28T23:15:00.000Z  5e9d0d95eda69955f709d1eb
4     RazakSat  2009-07-13T03:35:00.000Z  5e9d0d95eda69955f709d1eb


,name,date_utc,rocket
0,FalconSat,2006-03-24T22:30:00.000Z,5e9d0d95eda69955f709d1eb
1,DemoSat,2007-03-21T01:10:00.000Z,5e9d0d95eda69955f709d1eb
2,Trailblazer,2008-08-03T03:34:00.000Z,5e9d0d95eda69955f709d1eb
3,RatSat,2008-09-28T23:15:00.000Z,5e9d0d95eda69955f709d1eb
4,RazakSat,2009-07-13T03:35:00.000Z,5e9d0d95eda69955f709d1eb


### Select Falcon 9 flights with one core and one payload

In [2]:
falcon9_id = "5e9d0d95eda69973a809d1ec"
dates = pd.to_datetime(raw["date_utc"], utc=True).dt.strftime("%Y-%m-%d")
mask = (raw.rocket.eq(falcon9_id) & raw.cores.map(len).eq(1)
        & raw.payloads.map(len).eq(1) & dates.le("2020-11-13"))
launches = raw.loc[mask].sort_values("flight_number").reset_index(drop=True)
cores = pd.DataFrame(launches.cores.str[0].tolist())
flights = pd.DataFrame({
    "FlightNumber": range(1, len(launches) + 1),
    "Date": launches.date_utc.str[:10],
    "Flights": cores.flight,
    "Outcome": cores.landing_success.map(str) + " " + cores.landing_type.map(str),
    "GridFins": cores.gridfins, "Reused": cores.reused, "Legs": cores.legs,
    "LandingPad": cores.landpad,
})
flights.head()

Out[0]: 
   FlightNumber        Date  Flights  ... Reused   Legs  LandingPad
0             1  2010-06-04        1  ...  False  False        None
1             2  2012-05-22        1  ...  False  False        None
2             3  2013-03-01        1  ...  False  False        None
3             4  2013-09-29        1  ...  False  False        None
4             5  2013-12-03        1  ...  False  False        None

[5 rows x 8 columns]


,FlightNumber,Date,Flights,Outcome,GridFins,Reused,Legs,LandingPad
0,1,2010-06-04,1,None None,False,False,False,None
1,2,2012-05-22,1,None None,False,False,False,None
2,3,2013-03-01,1,None None,False,False,False,None
3,4,2013-09-29,1,False Ocean,False,False,False,None
4,5,2013-12-03,1,None None,False,False,False,None


### Join archived lookup fields and export
The course snapshot supplies the historical payload, orbit, launchpad and booster details. Its missing payload masses have already been mean-imputed. The launch date and sequential flight number together identify each record.

In [3]:
snapshot = pd.read_csv("data/dataset_part_1.csv")
keys = ["FlightNumber", "Date"]
lookup = [c for c in snapshot if c not in flights]
result = flights.merge(snapshot[keys + lookup], on=keys, validate="one_to_one")
assert len(result) == len(flights) == len(snapshot) == 90
result = result[snapshot.columns]
result["LandingPad"] = result["LandingPad"].fillna(float("nan"))
pd.testing.assert_frame_equal(result, snapshot, check_dtype=False)
result.to_csv("dataset_part_1.csv", index=False)
result.isna().sum()

Out[0]: 
FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64


FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

## Authors


<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD. 


<!--## Change Log
-->


<!--

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2020-09-20|1.1|Joseph|get result each time you run|
|2020-09-20|1.1|Azim |Created Part 1 Lab using SpaceX API|
|2020-09-20|1.0|Joseph |Modified Multiple Areas|
-->


Copyright ©IBM Corporation. All rights reserved.
